In [ ]:
import pandas as pd
import os 

# path for few_shot_and_zero_shot_comparison.csv 
data = pd.read_csv("")

groq_api_token = os.environ.get("GROQ_API_TOKEN")

qwen = "qwen/qwen3-32b"
llama = "llama-3.3-70b-versatile"

model = qwen


In [17]:
from groq import Groq

client = Groq(api_key=groq_api_token)

def judge(review: str, extraction_a: dict, extraction_b: dict) -> str:    
    prompt = f"""
You are evaluating two feature-centric opinion extraction systems.

TASK RULES:
- Extract tuples: (entity, feature, opinion, opinion_value)
- Entity, feature and opinion MUST exist in the original text (no generation)
- If feature is implicit, use "NULL"
- Opinion MUST be ≤ 5 words
- Opinion_value range: [-1, +1] where -1 = strongly negative, 0 = neutral, +1 = strongly positive

IMPORTANT:
- The order of A and B is random and does NOT indicate quality.
- More extracted tuples do NOT necessarily indicate better quality.
- Fewer extracted tuples do NOT necessarily indicate worse quality.
- Incorrect or hallucinated tuples must be penalized even if the extraction is more complete.
- Use TIE only if both extractions are equivalent in correctness and completeness.

PRIORITY OF EVALUATION:
1. Accuracy: entities, features and opinions truly exist in the text.
2. Rule correctness: proper use of NULL, opinion length ≤ 5 words, valid opinion_value.
3. Completeness: all opinions in the text are captured.

ORIGINAL TEXT:
{review}

EXTRACTION A:
{extraction_a}

EXTRACTION B:
{extraction_b}

First, internally verify each tuple in A and B against the original text.
Then decide which extraction better follows task rules and captures all opinions.

Respond with ONLY one word: A, B or TIE

Do NOT provide explanations.
"""

    # qwen requires reasoning_effort="none" to avoid reasoning steps

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are an objective judge. Respond ONLY with: A, B, or TIE"
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model=model,
        temperature=0.5,
        top_p=0.8,
        max_tokens=5,
        seed=42,
        reasoning_effort="none"
    )
    
    result = response.choices[0].message.content.strip().upper()
    if result not in ["A", "B", "TIE"]:
        print(f"WARNING: Invalid response from judge -> {result}")
    
    return result

In [18]:
results = {
    "A": 0,
    "B": 0,
    "TIE": 0,
}

In [19]:
import time
for index, row in data.iterrows():
    review = row['text']
    extraction_a = row['pred_tuples_zero_shot']
    extraction_b = row['pred_tuples_few_shot']

    decision = judge(review, extraction_a, extraction_b)
    results[decision] += 1
    
    time.sleep(1)

In [20]:
results

{'A': 64, 'B': 24, 'TIE': 12}